# Машинное обучение, ФКН ВШЭ

## Практическое домашнее задание 3.2  Продвинутая генерация признаков

### Общая информация

Дата выдачи: 23.02.2026

Мягкий дедлайн: 12.03.2026 23:59MSK

Жесткий дедлайн: 16.03.2026 23:59MSK

### О задании

В данном задании вы познакомитесь с менее тривиальными подходами для создания новых признаков в табличном машинном обучении. Вам понадобится подумать над тем, зачем мы делаем те или иные преобразования, научиться принимать решения, дающие наилучшие результаты, и узнать, как реализовывать их при помощи библиотек

### Оценивание и штрафы

См. базовую часть

### Формат сдачи
Задания сдаются через систему Anytask. Инвайт можно найти на странице курса. Присылать необходимо ноутбук с выполненным заданием. Сам ноутбук называйте в формате **homework-practice-03-advanced-Username.ipynb**, где Username — ваша фамилия.

### **Введение**

В этой части ноутбука задания посложнее дефолтного фит трансформа. Максимальная оценка за оба — 8 баллов, остальное вы можете получить, если примете участие в соревновании, и всего можете выбить аж 13 из 10. Тут ожидается больше самостоятельности, как от (почти) полноценной рабочей единицы: 
- Вы **сами** решаете, что <font color="#cb9255">**хотите**</font> делать. Пункты можно делать частично, можно скипать или сделать часть, **максимум баллов ограничен двумя**. **Посмотрите на все из них**, прежде чем приступать
- Вы **сами** чистите данные, если чуете в них подвох (теперь они далеко не такие няшные)
- Вы **сами** <font color="#f68c9d">**обосновываете**</font> (в голове, если не указано явно), могут они вам вообще помочь или нет (часть пунктов явно сильнее других)

Все прочие пожелания по тому, как строить графики, на чём фиттить, а на чём предиктить, сохраняются, будьте внимательны. Во всех пунктах с 📈 нужно добиться хотя бы минимального улучшения, относительно бейзлайна (того, что вышло в части **base**) чтобы получить балл (даже если улучшение на 0.005)

Ещё раз обратите внимание, что **максимум за advanced часть — 2 балла**, делать всё не нужно, только самое приятное. Мы в вас верим!

In [ ]:
import numpy as np
import optuna
import polars as pl

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.model_selection import TimeSeriesSplit


from pipeline import (
    combine_dfs,
    gen_objective,
    gen_objective_sgd,
    gen_objective_w_chat,
    get_objective_sgd_w_chat,
    get_tokenized_chats,
    gini,
    process_match_df,
    process_player_df,
)

In [ ]:
import plotly.express as px
from plotly.subplots import make_subplots

layout_dict = dict(
    margin=dict(l=20, r=20, t=40, b=20),
    width=600,
    height=400,
    paper_bgcolor="LightSteelBlue",
    title_font_size=14,
    xaxis_title_font_size=12,
    yaxis_title_font_size=12,
)

In [ ]:
df_train, ct = process_match_df()
df_test, _ = process_match_df(is_train=False, column_transformer=ct)
players = process_player_df()

In [ ]:
print(f"Train shape: {df_train.collect().shape}, test shape: {df_test.collect().shape}")

In [ ]:
df_train = combine_dfs(df_train, players)
df_test = combine_dfs(df_test, players)

In [ ]:
cols = (
    pl.selectors.starts_with("region_"),
    pl.selectors.starts_with("hero_"),
    "is_weekend",
    "avg_mmr_missing",
    "imputed_avg_mmr_log1p",
)
# this compiles the data in-memory, slow
X = df_train.select(*cols).collect()
y = df_train.select("radiant_win").collect()["radiant_win"]

objective = gen_objective(X, y)
objective_sgd = gen_objective_sgd(X, y)

In [ ]:
study_name = "simple_study"
sampler = optuna.samplers.TPESampler(seed=10)
study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    direction="maximize",
    pruner=None,
    storage=None,
)
# n_trials можно поставить и побольше, но пространство гиперпараметров здесь простое
study.optimize(objective, show_progress_bar=True, n_trials=3)

In [ ]:
study_name = "sgd_study"
sampler = optuna.samplers.TPESampler(seed=10)
# pruner = optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=1)
pruner = optuna.pruners.PatientPruner(
    optuna.pruners.HyperbandPruner(min_resource=1), patience=1
)
storage = f"sqlite:////data/{study_name}.db"

### if reset
try:
    optuna.delete_study(study_name=study_name, storage=storage)
except KeyError:
    pass  # Study didn't exist yet, which is fine

study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    direction="maximize",
    pruner=pruner,
    storage=storage,
    load_if_exists=True,
)
# n_trials можно поставить и побольше, но пространство гиперпараметров здесь простое
study.optimize(objective_sgd, show_progress_bar=False, n_trials=10)

### **Часть 4. Текста** (1.5 балла) <img align="center" src="https://static.wikia.nocookie.net/dota2_gamepedia/images/4/4f/Emoticon_blush.gif/revision/latest?cb=20180504011409">

В которой студент знакомится с внутренним миром дотеров

#### **Задание 4.1. Предобработка текста** (0.75 балла)

<span style="color:grey"><font size="1">Если вам когда-либо приходила в голову мысль, что создание Интернета было ошибкой, то, что ж, после этого задания сомнения могут отпасть.</font></span>

Для некоторых матчей имеется информация о том, что писали местные аборигены, в течение тех же **15 минут от начала матча**. К сожалению, доселе мы работали лишь с таблицами, а не с текстами, но не беда, простейшие подходы нейросетей не требуют, а в простых задачах, вроде бинарной классификации, работать будут не хуже

Откройте датафрейм `game_chat.csv`, выведите парочку текстов, посмотрите, как они устроены, как там хранятся множественные сообщения,  и так далее, что у нас есть, а чего, увы, нет

In [ ]:
# this processes chats using different tokenizers
get_tokenized_chats("nist")
get_tokenized_chats("toktok")
get_tokenized_chats("destructor")
get_tokenized_chats("tweet", force_recreate=False).sample(10)

In [ ]:
get_tokenized_chats("destructor", force_recreate=True)

Дём дальше. Тексты нужно готовить, прежде чем пихать их в модель. Оценивать выбросы здесь довольно проблематично, в силу специфики дотерских сообщений, хотя вы, конечно, можете попытаться. Речь здесь про базовую предобработку.

Задача минимум, тут мы вам поможем:
- Разобраться с библиотекой `nltk` и разбить текст на токены — отдельные сущности, составляющие текст (чаще всего слова, но бывает и что-то другое, надо разобраться). Как бить текст — вопрос неоднозначный. В целом подойдёт любой способ, но какие-то [токенизаторы](https://www.nltk.org/api/nltk.tokenize.html) могут сразу покрыть часть проблем с текстами в задаче максимум
- Лемматизировать текст (привести слова к начальной форме). <br>
<font color="#cb9255">**Варианта два**</font>: манкипатчить [`pymorphy2`](https://pymorphy2.readthedocs.io/en/stable/) или откатывать версии (там вылезет ошибка, если у вас слишком новый питон), либо применять [`mystem`](https://pypi.org/project/pymystem3/), что может затянуться на несколько часов, зато лемматизация будет точнее

Задача максимум, тут вам нужно понять, как всё обработать, самим (включать-не включать, выкинуть-оставить — валидны **все** варианты, но только, если, есть, <font color="#f68c9d">**обоснование**</font>):
- повторы символов
- знаки препинания
- стоп-слова
- нижний регистр
  
Вам нужно пройтись по всем пунктам, не обязательно в этом порядке. Если найдёте что-то ещё — круто, молодцы, можно тоже пофиксить

<div style="border-left: 5px solid #ff748c; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** ну что, как обрабатываем текст?

**Ответ:**

</div>

In [ ]:
objective_with_chat = gen_objective_w_chat(
    df_train.select("match_id", "radiant_win").collect(),
    df_train.select("radiant_win").collect().to_series().shuffle(123),
)

study_name = "chat_study"
storage = f"sqlite:////data/{study_name}.db"
sampler = optuna.samplers.TPESampler(seed=10)

# if reset
try:
    optuna.delete_study(study_name=study_name, storage=storage)
except KeyError:
    pass  # Study didn't exist yet, which is fine

study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    direction="maximize",
    pruner=None,
    storage=storage,
    load_if_exists=True,
)
# n_trials можно поставить и побольше, но пространство гиперпараметров здесь простое
study.optimize(objective_with_chat, show_progress_bar=True, n_trials=20)

In [ ]:
objective_sgd_w_chat = get_objective_sgd_w_chat(
    df_train.select("match_id", "radiant_win").collect(),
    df_train.select("radiant_win").collect().to_series(),
)

study_name = "sgd_chat_study"
sampler = optuna.samplers.TPESampler(seed=10)
# pruner = optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=1)
pruner = optuna.pruners.PatientPruner(
    optuna.pruners.HyperbandPruner(min_resource=1), patience=1
)

study = optuna.create_study(
    study_name=study_name,
    sampler=sampler,
    direction="maximize",
    pruner=pruner,
    storage=None,
)
# n_trials можно поставить и побольше, но пространство гиперпараметров здесь простое
study.optimize(objective_sgd_w_chat, show_progress_bar=True, n_trials=2)

In [ ]:
# reconstructing word only model
from sklearn.svm import LinearSVC

from pipeline import adjoin_chat, learn_chat_embedding

# {'C': 46.654739937316776, 'max_iter': 95, 'loss': 'squared_hinge', 'penalty': 'l1', 'tokenizer_name': 'destructor', 'model': 'tfidf', 'ngram_max': 1, 'min_df': 50, 'max_features': 1000}

msgs = get_tokenized_chats("destructor")
X = (
    df_train.select("match_id", "radiant_win")
    .collect()
    .join(msgs, on="match_id", how="inner")
    .with_columns(
        pl.col("radiant_chat_norm").fill_null(pl.lit([])),
        pl.col("dire_chat_norm").fill_null(pl.lit([])),
    )
)
y = X.get_column("radiant_win")
X = X.drop("radiant_win")

chat_vec = learn_chat_embedding(
    msgs, {"model": "tfidf", "ngram_range": (1, 1), "min_df": 50, "max_features": 1000}
)

model = LinearSVC(
    **{
        "C": 46.654739937316776,
        "max_iter": 195,
        "loss": "squared_hinge",
        "penalty": "l1",
    }
)
cv_time = TimeSeriesSplit(n_splits=4)
y_hat = np.concatenate(
    [
        model.fit(
            adjoin_chat(X[tr], chat_vec, mode="both").drop("match_id"),
            y[tr],
        ).decision_function(adjoin_chat(X[ts], chat_vec, mode="both").drop("match_id"))
        for tr, ts in cv_time.split(X)
    ]
)

gini(y[-len(y_hat) :], y_hat)

In [ ]:
tr, ts = list(cv_time.split(X))[-1]
model.fit(
    adjoin_chat(X[tr], chat_vec, mode="both").drop("match_id"),
    y[tr],
)
chat_sample = (
    X[ts]
    .with_columns(
        pred=model.decision_function(
            adjoin_chat(X[ts], chat_vec, mode="both").drop("match_id")
        )
    )
    .sort("pred")
)

In [ ]:
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(400)

pl.scan_csv(
    f"/data/ml-course-hse/ml1-2026-spring/homework-practice-03-features/game_chat.csv",
    try_parse_dates=True,
).filter(pl.col("match_id").is_in(chat_sample[-40:, "match_id"].to_list())).collect()

#### 📈 **Задание 4.2. Векторизация** (0.5 балла)

Ура, если у вас получились токены, то наконец-то можно что-то закодировать, но как? Рад, что вы спросили. Читайте конспект семинаров или документацию

<table width="800" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="50%">
      <font color="#cb9255">CountVectorizer</font>
    </th>
    <th width="50%">
      <font color="#cb9255">TfIdfVectorizer</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Берёт и считает, сколько раз в тексте <br>
      встретилось то или иное слово. Похож <br>
      на наш энкодер из части 2.
      <br><br>
      <table border="1" cellpadding="4" cellspacing="0">
        <thead>
          <tr>
            <th scope="col">word_1</th>
            <th scope="col">word_2</th>
            <th scope="col">word_3</th>
            <th scope="col">...</th>
            <th scope="col">word_m</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td>1</td>
            <td>0</td>
            <td>2</td>
            <td>...</td>
            <td>100</td>
          </tr>
        </tbody>
      </table>
    </td>
    <td valign="top">
      Более хитрая штука: вместе с количеством слов (tf)<br>
      считает их важность (idf). Слова, встречающиеся <br>
      во всех документах, считаются не важными<br>
      и зануляются (idf = log1).
      <br><br>
      <table border="1" cellpadding="4" cellspacing="0">
        <thead>
          <tr>
            <th scope="col"></th>
            <th scope="col">word_1</th>
            <th scope="col">word_2</th>
            <th scope="col">word_3</th>
            <th scope="col">...</th>
            <th scope="col">word_m</th>
          </tr>
        </thead>
        <tbody>
          <tr>
            <td><b>text_1</b></td>
            <td>1*log2</td>
            <td>0</td>
            <td>2*log2</td>
            <td>...</td>
            <td>100*log1</td>
          </tr>
          <tr>
            <td><b>text_2</b></td>
            <td>0</td>
            <td>10*log2</td>
            <td>0</td>
            <td>...</td>
            <td>100*log1</td>
          </tr>
        </tbody>
      </table>
    </td>
  </tr>
</table>


Оба векторайзера хороши, но у каждого из них есть <font color="#cb9255">**гиперпараметры**</font>. Естественно, они повлияют на качество, <font color="#cb9255">**можете подобрать их**</font> попозже, дефолтные тоже должны показать эффект

Обучите по векторайзеру на чатах Radiant и Dire. Приклейте результат к вашему датасету и обучите модель на всём получившемся великолепии (sparse формат убирать не рекомендуется)

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 4.3. Визуализация** (0.25 балла)

Для любителей графиков есть малюсенькое задание: визуализируйте облако слов с наибольшими по модулю весами (разделите их на условно *"позитивные"* и *"негативные"*)

In [ ]:
fig = px.bar(
    pl.DataFrame({"name": model.feature_names_in_, "value": model.coef_[0]})
    .sort("value")
    .filter(pl.col("value").abs() > 0.2),
    x="name",
    y="value",
)

fig.update_layout(**layout_dict)
fig.update_layout({"width": 2000, "height": 600})

<div style="border-left: 5px solid #ff748c; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** что думаете? Сейчас и вообще по модели — негативные получились слова или так, едва?

**Ответ:**

</div>

### **Часть 5. Агрегации** (1.75 балла) <img align="center" src="https://static.wikia.nocookie.net/dota2_gamepedia/images/4/4a/Techies_emoticon.gif/revision/latest?cb=20180504014918">

В которой студент начинает ведать

#### 📈 **Задание 5.1. Статистики матча** (0.75 балла)

Есть у нас в данных большой кусок про advantage — преимущество команды сил Света с точностью до минуты, по золоту и опыту, всё так же **в пределах 15 минут**. Лежат они в `dota_adv.csv`. Чем больше число, тем больше шанс на победу — всё просто. Только график, как правило, не линеен.

С ними в очередной раз есть *нюансы* — необходимо разобраться, как они там лежат, и всё ли там в порядке со значениями, но это меньшая из проблем. А также нарисовать парочку advantage, чтобы было понимание, как они себя ведут

In [ ]:
from pipeline import (
    adjoin_adv_stats,
    bin_cols_to_scale,
    clip_cols_mad,
    clip_cols_std,
    ColumnNormalizer,
)

cols_adv = ("radiant_gold_adv", "radiant_exp_adv")
adv = (
    pl.scan_csv(
        f"/data/ml-course-hse/ml1-2026-spring/homework-practice-03-features/dota_adv.csv",
        try_parse_dates=True,
    )
    .filter(pl.col("radiant_gold_adv") != "[]")
    .with_columns(
        pl.col(cols_adv)
        .str.replace_all(r"[\[\]]", "")
        .str.replace_all(r"\s{2,}", " ")
        .str.strip_chars()
        .str.split(" ")
    )
    .explode(cols_adv)
    .with_columns(
        pl.col(cols_adv).cast(pl.Float64),
        pl.row_index("t").over("match_id"),
    )
    .with_columns(
        radiant_gold_adv=pl.when(pl.col.radiant_gold_adv.abs() >= 5e5)
        .then(None)
        .otherwise(pl.col.radiant_gold_adv),
        radiant_exp_adv=pl.when(pl.col.radiant_exp_adv.abs() >= 5e5)
        .then(None)
        .otherwise(pl.col.radiant_exp_adv),
    )
    .with_columns(
        pl.col("radiant_gold_adv", "radiant_exp_adv").fill_null(0.0),
        pl.col("radiant_gold_adv", "radiant_exp_adv").is_null().name.suffix("_missing"),
    )
).collect()
# each group has exactly 15 values (not sure what happens with shorter games)
# most groups have nulls
# about 19% of data are duplicates but it doesn't affect the fit materially
# adv = adv.sort('match_id', 't').filter((pl.col.radiant_gold_adv != pl.col.radiant_gold_adv.shift().over('match_id')) | (pl.col.radiant_exp_adv != pl.col.radiant_exp_adv.shift().over('match_id')))

# adv_sample = adv.collect()[:1000]

In [ ]:
# aggregating with different normalizations (for plotting, but I'll need a pipeline for this to support fitting)
adv = clip_cols_mad(
    clip_cols_std(adv, cols_adv, group_cols=["t"]), cols_adv, group_cols=["t"]
)
adv = adv.with_columns(
    (pl.col(cols_adv).rank().over("t") / pl.len().over("t")).name.suffix("_q"),
    (
        (
            pl.col([f"{c}_std_clip" for c in cols_adv])
            - pl.col([f"{c}_std_clip" for c in cols_adv]).mean().over("t")
        )
        / pl.col([f"{c}_std_clip" for c in cols_adv]).std().over("t")
    )
    .name.replace("_std_clip", "_l2n")
    .fill_nan(0.0),
    (
        pl.col([f"{c}_mad_clip" for c in cols_adv])
        / (
            (
                pl.col([f"{c}_mad_clip" for c in cols_adv])
                - pl.col([f"{c}_mad_clip" for c in cols_adv]).median().over("t")
            )
            .abs()
            .median()
            .over("t")
        )
    )
    .name.replace("_mad_clip", "_l1n")
    .fill_nan(0.0),
)

In [ ]:
pl.Config.set_tbl_rows(20)

adv.group_by("t").agg(
    pl.col(cols_adv).mean().name.suffix("_mean"),
    pl.col(cols_adv).mean().name.suffix("_median"),
    pl.col(cols_adv).abs().mean().name.suffix("_abs_mean"),
    pl.col(cols_adv).std().name.suffix("_std"),
    (pl.col(cols_adv) - pl.col(cols_adv).mean()).abs().mean().name.suffix("_mad"),
).sort("t")

In [ ]:
pl.Config.set_fmt_table_cell_list_len(20)
adv.select(
    pl.col(cols_adv)
    .quantile(
        [
            0.0001,
            0.001,
            0.01,
            0.05,
            0.1,
            0.25,
            0.5,
            0.75,
            0.9,
            0.95,
            0.99,
            0.999,
            0.9999,
        ]
    )
    .name.suffix("_q"),
    pl.col(cols_adv).min().name.suffix("_min"),
    pl.col(cols_adv).max().name.suffix("_max"),
)

Для начала возьмём простые агрегации. Можете взять те, что вам знакомы (статистики - среднее, стд и др.), можете взять фан факты в вашей любимой библиотеке для данных, например [тут](https://pandas.pydata.org/docs/user_guide/groupby.html#aggregation) или [тут](https://docs.pola.rs/api/python/stable/reference/expressions/aggregation.html).

Задание:
- взять 4 статистики из библиотеки, применить к обеим колонкам `_adv`, <font color="#f68c9d">**обдумать**</font>, почему именно они
- одну из статистику выше разбить по командам, и точно так же примените к колонкам (получится что-то типа `agg_xp` -> `agg_dire_xp`, `agg_radiant_xp`)

<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Тут мы встаём на скользкую дорожку переобучения. Агрегаций можно сделать **очень** много. Добавьте их все, и ваша модель превратится в тыкву. Удобнее будет сразу бить их на группы, например `features_last`, `features_q25`, `features_kurtosis_dire_10min+` и так далее, в зависимости от степени упоротости

C другой стороны, агрегации это самая сильная группа фичей, и для десяточки лучше целиться именно в них

</div>

<div style="border-left: 5px solid #f68c9d; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** какие агрегации берём?

**Ответ:**

</div>

In [ ]:
# similar pattern of increasing over time
fig = px.line(
    adv.filter(
        pl.col.match_id.is_in(adv.select("match_id").to_series().sample(250).implode())
    ).select("t", "match_id", "radiant_exp_adv"),
    x="t",
    y="radiant_exp_adv",
    line_group="match_id",
    # color = 'gray'
    # opacity =.5,
)

fig.update_traces(line=dict(color="grey"), opacity=0.2)
fig.update_layout(**layout_dict)

In [ ]:
# looks like the "trend" should be last few entries increase
fig = px.line(
    adv.filter(
        pl.col.t > 0,
        pl.col.match_id.is_in(adv.select("match_id").to_series().sample(20).implode()),
    ).select("t", "match_id", "radiant_gold_adv_l2n"),
    x="t",
    y="radiant_gold_adv_l2n",
    line_group="match_id",
)

fig.update_traces(line=dict(color="grey"), opacity=0.2)
fig.update_layout(**layout_dict)

In [ ]:
fig = px.histogram(
    # a.filter(pl.col.t == 10).select("radiant_gold_adv_q").collect(),
    adv.filter(pl.col.t == 10).unpivot(
        ["radiant_gold_adv_q", "radiant_gold_adv_l1n", "radiant_gold_adv_l2n"],
        index="match_id",
        variable_name="feature",
    ),
    x="value",
    facet_row="feature",
    nbins=100,
)

fig.update_layout(**layout_dict)
fig.update_layout(height=800)

In [ ]:
m = adjoin_adv_stats(df_train.select("match_id", "radiant_win").collect(), adv)

In [ ]:
# I.1 last
nbins = 10
bin_cols = (
    [f"{c}_std_clip_last" for c in cols_adv]
    + [f"{c}_q_last" for c in cols_adv]
    + [f"{c}_l2n_last" for c in cols_adv]
)

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log())
    .sort("feature"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
)

fig.update_layout(**layout_dict)
fig.update_xaxes(matches=None)
fig.update_layout(height=1600)

In [ ]:
# I.2 mean
nbins = 10
bin_cols = (
    [f"{c}_std_clip_avg" for c in cols_adv]
    + [f"{c}_q_avg" for c in cols_adv]
    + [f"{c}_l2n_avg" for c in cols_adv]
)

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log())
    .sort("feature"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
)

fig.update_layout(**layout_dict)
fig.update_xaxes(matches=None)
fig.update_layout(height=1600)

In [ ]:
# II.1 groups of 5s
nbins = 10
bin_cols = [c for c in m.schema.names() if "_t5_" in c and "std_clip" not in c]

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(
        radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log(),
        feature=pl.col.feature.str.replace(r"_t5_.*", ""),
        tgp=pl.col.feature.str.extract(r"_t5_(.*)", 1).cast(pl.Int64),
    )
    .sort("feature", "tgp"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
    facet_col="tgp",
)

fig.update_layout(**layout_dict)
# fig.update_xaxes(matches=None)
fig.update_layout(height=1600, width=1200)

In [ ]:
# II.2 groups of 3s
nbins = 10
bin_cols = [c for c in m.schema.names() if "_t3_" in c and "std_clip" in c]

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(
        radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log(),
        feature=pl.col.feature.str.replace(r"_t3_.*", ""),
        tgp=pl.col.feature.str.extract(r"_t3_(.*)", 1).cast(pl.Int64),
    )
    .sort("feature", "tgp"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
    facet_col="tgp",
)

fig.update_layout(**layout_dict)
# fig.update_xaxes(matches=None)
fig.update_layout(height=1600, width=2000)

In [ ]:
# III.1 groups of 5s
nbins = 10
bin_cols = [c for c in m.schema.names() if c.endswith("_t")]

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(
        radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log(),
        feature=pl.col.feature.str.extract(r"_(.*)", 1),
        tgp=pl.col.feature.str.extract(r"^([^_]+)", 1),
    )
    .sort("feature", "tgp"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
    facet_col="tgp",
)

fig.update_layout(**layout_dict)
fig.update_xaxes(matches=None)
fig.update_layout(height=1200, width=1200)

In [ ]:
# III.2 groups of 5s
nbins = 10
bin_cols = [c for c in m.schema.names() if c.endswith("_t_last5")]

fig = px.scatter(
    m.unpivot(bin_cols, index=["match_id", "radiant_win"], variable_name="feature")
    .join(
        bin_cols_to_scale(m, bin_cols, nbins=nbins)
        .unpivot(
            pl.selectors.ends_with(f"{nbins}bin"),
            index="match_id",
            variable_name="feature",
            value_name="bin",
        )
        .with_columns(pl.col.feature.str.replace(f"_{nbins}bin", "")),
        on=["match_id", "feature"],
        how="inner",
    )
    .group_by("feature", "bin")
    .agg(pl.col("value", "radiant_win").mean(), ct=pl.len().log())
    .with_columns(
        radiant_win=(pl.col.radiant_win / (1.0 - pl.col.radiant_win)).log(),
        feature=pl.col.feature.str.extract(r"_(.*)", 1),
        tgp=pl.col.feature.str.extract(r"^([^_]+)", 1),
    )
    .sort("feature", "tgp"),
    x="value",
    y="radiant_win",
    size="ct",
    facet_row="feature",
    facet_col="tgp",
)

fig.update_layout(**layout_dict)
fig.update_xaxes(matches=None)
fig.update_layout(height=1200, width=1200)

Обучите модель по агрегациям (одной группе или нескольким) + предыдущим фичам. Чтобы получить фулл балл, придётся показать, что хотя бы минимальный импрув есть, относительно бейзлайна

In [ ]:
fold_cache = {}

In [ ]:
# starting with df_train and adv. Assume that adv only has "_std_clip" columns (meanimal leakage)
X = (
    df_train.select("match_id", "radiant_win")
    .filter(pl.col.match_id.is_in(adv.get_column("match_id").implode()))
    .collect()
)
y = X.get_column("radiant_win")
X = X.drop("radiant_win")

cv_time = TimeSeriesSplit(n_splits=4)
offset = len(next(cv_time.split(X))[0])
y_hat = pl.zeros(len(y), dtype=pl.Float64, eager=True)

for i, (tr, ts) in enumerate(cv_time.split(X)):
    if i in fold_cache:
        Xn = fold_cache[i]
    else:
        cn = ColumnNormalizer(
            ["radiant_gold_adv_std_clip", "radiant_exp_adv_std_clip"], ["t"]
        )
        cn.fit(
            adv.filter(
                ~pl.col.radiant_gold_adv_missing,
                ~pl.col.radiant_exp_adv_missing,
                pl.col.match_id.is_in(X[tr].get_column("match_id").implode()),
            )
        )
        advn = (
            cn.transform(
                adv.select(
                    "match_id",
                    "t",
                    "radiant_gold_adv_std_clip",
                    "radiant_exp_adv_std_clip",
                )
            )
            .rename(
                {
                    "radiant_gold_adv_std_clip_l2n": "radiant_gold_adv_l2n",
                    "radiant_gold_adv_std_clip_q": "radiant_gold_adv_q",
                    "radiant_exp_adv_std_clip_l2n": "radiant_exp_adv_l2n",
                    "radiant_exp_adv_std_clip_q": "radiant_exp_adv_q",
                }
            )
            .with_columns(pl.selectors.starts_with("radiant_").fill_nan(0.0))
        )
        Xn = adjoin_adv_stats(X, advn)
        fold_cache[i] = Xn

    # cols = [c for c in Xn.columns if c.endswith("exp_adv_q_last")]
    # cols = [c for c in Xn.columns if 'gold' in c and '_beta_' in c and '_clip_' in c]
    # cols = [c for c in Xn.columns if 'radiant_gold_' in c and '_clip_' in c and c.endswith('_t_last5')]
    cols = [
        "beta_radiant_gold_adv_std_clip_exp_t",
        "beta_radiant_exp_adv_std_clip_exp_t",
        "alpha_radiant_gold_adv_std_clip_exp_t",
        "alpha_radiant_exp_adv_std_clip_exp_t",
    ]
    cols += ["radiant_gold_adv_std_clip_last", "radiant_exp_adv_std_clip_last"]
    model = LogisticRegression(C=1e3)
    model.fit(Xn[tr, cols].with_columns(pl.all().fill_nan(0.0)), y[tr])
    y_hat[ts] = model.decision_function(
        Xn[ts, cols].with_columns(pl.all().fill_nan(0.0))
    )

print(gini(y[-len(y_hat) :], y_hat), cols)

In [ ]:
# gini results
0.3663487252563902 ['radiant_gold_adv_std_clip_avg']  0.32843062213190355 ['radiant_gold_adv_q_avg'] 0.3289096011815116 ['radiant_gold_adv_l2n_avg']
0.2565856771355255 ['radiant_exp_adv_std_clip_avg'] 0.2510090871795141 ['radiant_exp_adv_l2n_avg'] 0.2509971837679632 ['radiant_exp_adv_q_avg']
0.3777431213327809 ['radiant_gold_adv_std_clip_avg', 'radiant_exp_adv_std_clip_avg'] 0.3331073644669056 ['radiant_gold_adv_l2n_avg', 'radiant_exp_adv_l2n_avg'] 0.3318689398248038 ['radiant_gold_adv_q_avg', 'radiant_exp_adv_q_avg']

0.3749363515651305 ['radiant_gold_adv_std_clip_last'] 0.3749434178489639 ['radiant_gold_adv_l2n_last'] 0.3755106595875959 ['radiant_gold_adv_q_last']
0.2359188144983635 ['radiant_exp_adv_std_clip_last'] 0.23592450961829492 ['radiant_exp_adv_l2n_last'] 0.23679066102847868 ['radiant_exp_adv_q_last']
0.4169941730512645 ['radiant_gold_adv_std_clip_last', 'radiant_exp_adv_std_clip_last'] 0.41700543522652955 ['radiant_gold_adv_l2n_last', 'radiant_exp_adv_l2n_last'] 0.41680276559277774 ['radiant_gold_adv_q_last', 'radiant_exp_adv_q_last']

0.3860903159996609 ['radiant_gold_adv_std_clip_t3_0', 'radiant_gold_adv_std_clip_t3_3', 'radiant_gold_adv_std_clip_t3_6', 'radiant_gold_adv_std_clip_t3_9', 'radiant_gold_adv_std_clip_t3_12']
0.2729770204735904 ['radiant_exp_adv_std_clip_t3_0', 'radiant_exp_adv_std_clip_t3_3', 'radiant_exp_adv_std_clip_t3_6', 'radiant_exp_adv_std_clip_t3_9', 'radiant_exp_adv_std_clip_t3_12']
0.4150703683983401 ['radiant_gold_adv_std_clip_t3_0', 'radiant_gold_adv_std_clip_t3_3', 'radiant_gold_adv_std_clip_t3_6', 'radiant_gold_adv_std_clip_t3_9', 'radiant_gold_adv_std_clip_t3_12', 'radiant_exp_adv_std_clip_t3_0', 'radiant_exp_adv_std_clip_t3_3', 'radiant_exp_adv_std_clip_t3_6', 'radiant_exp_adv_std_clip_t3_9', 'radiant_exp_adv_std_clip_t3_12']

0.3792222830640011 ['beta_radiant_gold_adv_std_clip_exp_t', 'r2_radiant_gold_adv_std_clip_exp_t', 'alpha_radiant_gold_adv_std_clip_exp_t']
0.2570512440061983 ['beta_radiant_exp_adv_std_clip_exp_t', 'r2_radiant_exp_adv_std_clip_exp_t', 'alpha_radiant_exp_adv_std_clip_exp_t']
0.402368242711864 ['beta_radiant_gold_adv_std_clip_exp_t', 'beta_radiant_exp_adv_std_clip_exp_t', 'r2_radiant_gold_adv_std_clip_exp_t', 'r2_radiant_exp_adv_std_clip_exp_t', 'alpha_radiant_gold_adv_std_clip_exp_t', 'alpha_radiant_exp_adv_std_clip_exp_t']

0.37247983218064395 ['beta_radiant_gold_adv_l2n_t', 'r2_radiant_gold_adv_l2n_t', 'alpha_radiant_gold_adv_l2n_t']
0.2536803008749091 ['beta_radiant_exp_adv_l2n_t', 'r2_radiant_exp_adv_l2n_t', 'alpha_radiant_exp_adv_l2n_t']
0.37238158532391785 ['beta_radiant_gold_adv_q_t', 'r2_radiant_gold_adv_q_t', 'alpha_radiant_gold_adv_q_t']
0.2542927522579066 ['beta_radiant_exp_adv_q_t', 'r2_radiant_exp_adv_q_t', 'alpha_radiant_exp_adv_q_t']

0.37679832384979184 ['beta_radiant_gold_adv_std_clip_exp_t_last5', 'r2_radiant_gold_adv_std_clip_exp_t_last5', 'alpha_radiant_gold_adv_std_clip_exp_t_last5']
0.23529836049869313 ['beta_radiant_exp_adv_std_clip_exp_t_last5', 'r2_radiant_exp_adv_std_clip_exp_t_last5', 'alpha_radiant_exp_adv_std_clip_exp_t_last5']


0.42629153721672375 ['beta_radiant_gold_adv_std_clip_exp_t', 'beta_radiant_exp_adv_std_clip_exp_t', 'r2_radiant_gold_adv_std_clip_exp_t', 'r2_radiant_exp_adv_std_clip_exp_t', 'alpha_radiant_gold_adv_std_clip_exp_t', 'alpha_radiant_exp_adv_std_clip_exp_t', 'radiant_gold_adv_l2n_last', 'radiant_exp_adv_l2n_last']

# final: alpha beta and last value. no need for quantiles
0.4257761695554134 ['beta_radiant_gold_adv_std_clip_exp_t', 'beta_radiant_exp_adv_std_clip_exp_t', 'alpha_radiant_gold_adv_std_clip_exp_t', 'alpha_radiant_exp_adv_std_clip_exp_t', 'radiant_gold_adv_std_clip_last', 'radiant_exp_adv_std_clip_last']



#### 📈 **Задание 5.2. Тренд** (0.5 балла)

Каждый уважающий себя лудоман знает, что 99% процентов игроков останавливается ровно перед тем, как сорвать джекпот. Так и здесь — если команда с треском проигрывает в первые 15 минут матча, возможно это признак камбека в следующие 50, как знать? Попробуем собрать агрегацию похитрее — она будет обозначать тренд, который есть в графиках преимущества, и если пословица верна, наша модель уловит эту зависимость.

<span style="color:grey"><font size="1">Администрация курса МО-1 категорически против азартных игр, пример приводится сугубо в образовательных целях.</font></span>

<div style="border-left: 5px solid #f68c9d; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(255, 116, 140, 0.05);">

**Вопрос:** для чего нам вообще тренд? Полезная ли это агрегация?

**Ответ:**

</div>

Агрегировать можно и вещи несколько более прикольные, чем те, что есть в основном функционале. Делать это мы будем, как вы наверняка догадались, трансформером, ну а чем же ещё. Что он умеет?

1. Принимает на вход функцию колонку и <font color="#cb9255">**параметры**</font> на ваш вкус, как минимум `method`, метод расчёта `slope`
2. Выделяет коэффициент наклона (`slope`, он же $\alpha$) при помощи одного из методов:
   - `'delta'`: разность первого и последнего значений $|x_{\max} - x_{\min}|$
   - `'OLS'`: линейная регрессия, обученная методом МНК $(X^TX)^{-1}X^Ty$
   - альтернативный метод, порождённый вашей бурной фантазией
3. Считает `r2` и `intercept` для одного advantage (если что это тоже могут быть наши фичи!)

In [ ]:
from typing import Iterable


class TrendTransformer:

    def __init__(self, columns: Iterable[str]):
        self.columns = columns

    def fit(self, X, y=None):
        pass

    def transform(self, X, y=None):
        # ヾ(⌐■_■)ノ♪ your code here
        raise Exception("transform method not implemented")

Реализуйте трансформер. Критерий успеха, вновь, качество — фича должна помочь, хотя бы на долю пункта

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **📈 Задание 5.3. Бинаризация** (0.5 балла)

Ровно одну прикольную фишку для числовых признаков мы пока что не рассмотрели — бинаризацию. Если вы до неё уже догадались, то вы — гений, не думали на <font color="cb9255">**МОП**</font>? А если нет, суть такова:

1. Берём отрезок advantage и бьём его на несколько бинов
2. Бины можно использовать, как фичу саму по себе, а можно подсобрать внутри неё агрегации

<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Во-первых, ваша модель внезапно становится нелинейной, хоть и кусочной, это полный отвал \
Во-вторых, это простейший пример ансамбля, если бинаризовать таргет (но у нас, увы нет смысла, он дискретный). Нелинейность полезна почему — в первые минуты преимущество не так решает, как в последние. \
В-третьих, это фильтрует шумный сигнал, выбросы то отлетят в соответствующий бин

</div>

Попробуем? Бинаризуйте признаки advantage: занумеруйте их (сделайте категорию) и посчитайте побиновые агрегации

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

### **Часть 6. Около ML** (2 балла) <img height=25px width=35px align="center" src="https://media1.tenor.com/m/72ScVNgTGpYAAAAC/kaneki-tokyo-ghoul.gif"></img>

В которой студент жесточайше чиллит после пережитого ужаса

#### **Задание 6.1. Пайплайн** (0.5 балла)

Работать в ноутбуках становится экспоненциально тяжелее по мере разрастания модели. Чтобы немножко упорядочить хаос, вам предлагается засунуть всё в один пайплайн. Критерии:

- функция или класс (может понравиться `ColumnTransformer` и `Pipeline`)
- возможность нажать одну кнопку, чтобы запустить пайплайн, уйти пить пиво и вернуться к уже готовому submission для Kaggle
- возможность передать флаги (какие фичи добавляем) и параметры (если есть разные варианты сбора параметров)
- включает в себя все пункты, к которым вы прикоснулись в рамках домашнего задания

А вот как именно это делать — дело ваше, для себя же стараетесь

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.2. Storage** (0.25 балла)

Вдогоночку можно ещё и создать псевдо-БД, чтобы хранить наши шедевры и не потеряться в тысячах моделек. Давайте вот такую штуку запилим:

- датафрейм или честная БД для версионирования моделей
- для каждой модели есть уникальный идентификатор
- для каждой модели сохраняются её гиперпараметры или параметры всего пайплайна (если вы его сделали)
- для каждой модели хранятся метрики на валидации

Сделайте и продемонстрируйте

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.3. CuML** (0.25 балла)

Если вы таки осмелились делать домашку именно на Kaggle, то поздравляю, пожалуй, это самое здравое решение в этой дз. Чтобы использовать его возможности по полной, пересядьте с вашей модели из `sklearn`, которую вы выбрали в задании про даты **(1.3)**, на модель из `cuml`. 

[Разберитесь](https://docs.rapids.ai/api/cuml/stable/), как они используют GPU и проведите тест-драйв на любом наборе фичей

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.4. High tech. Low life** (0.5 балла)

Если вы следите за новостями, то, наверное, заметили появление хайповейших LLM. Злые языки утверждают, что обычному работяге фит предиктору не место в мире будущего, и его заменит ИИ. Давайте в этом (раз)убедимся.

Попробуйте:
1. Спросить у вашей любимой нейросети, какие признаки она может для вас придумать. Можете опираться на пункты выше, можете придумать что-то свое. Но помните, что как говорится, какой стол, такой и стул, поэтому пишите промпты с умом.
2. Показать, что нейросеть вам посоветовала, и реализовать это
3. Проанализировать результат и сделать решительный вывод, хуже ли вы, чем языковая модель.

Попытайтесь либо вспомнить, либо посмотреть, что у нас ещё есть в данных. Там достаточно много полезной информации, которую мы либо совсем никак не брали, либо брали, но поверхностно, либо брали, но можно сделать ещё круче, старые пункты тоже можно доработать

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

#### **Задание 6.5. Отбор признаков** (0.5 балла)

Когда признаков становится так много, что ваша оперативка начинает рыдать, а модель переобучается, как чёрт, поможет только одно средство — отбор фичей!
Это первое и единственное задание, в котором <font color="#cb9255">**выбора**</font> аж три:

<table width="100%" border="1" cellpadding="8" cellspacing="0">
  <tr>
    <th width="33%">
      <font color="#cb9255">Sequential Feature Selector</font>
    </th>
    <th width="33%">
      <font color="#cb9255">Greedy Selection</font>
    </th>
    <th width="33%">
      <font color="#cb9255">Recursive Feature Elimination</font>
    </th>
  </tr>
  <tr>
    <td valign="top">
      Делаем итеративно. На каждой итерации <br>
      оцениваем важность фичей по их <br>
      импортансу (<code>coef_</code>), берём <br>
      топ‑n худших, выкидываем, go to 0.
    </td>
    <td valign="top">
      Перебираем все комбинации признаков <br>
      и выбираем наилучшую. Звучит тупо, <br>
      но комбинации можно брать по группам <br>
      (например, тексты, агрегации средних <br>
      и т.д.), тогда это не так долго <br>
      <b>(2 часа на 200 признаков)</b>.
    </td>
    <td valign="top">
      Идём с конца и выкидываем по признаку. <br>
      На каждом шаге обучаем по одной модели <br>
      без одного признака (обучаем d‑1 моделей), <br>
      выбираем из них худшую — такой признак <br>
      и устраняем.
    </td>
  </tr>
  <tr>
    <td valign="top">
      Быстро <b>(около 20 минут <br>
      на 200 фичах)</b>, но веса линейной <br>
      регрессии плохо оценивают важность <br>
      фичей; это лучше работает для <br>
      сильных моделей.
    </td>
    <td valign="top">
      Не теряем интеракции. Баланс <br>
      скорость–качество.
    </td>
    <td valign="top">
      Возмутительно долго <b>(10 часов <br>
      на 200 признаков)</b>, но гарантирует <br>
      минимальные потери в качестве.
    </td>
  </tr>
</table>


<div style="border-left: 5px solid #647cb8; padding: 10px 20px 2px; margin: 15px 0 15px 20px; max-width: 800px; background-color: rgba(95, 121, 179, 0.05);">

Это улучшит качество, если вы уже страдаете от миллиарда малополезных фичей. Но для получения балла это не нужно, только верный алгоритм

</div>

Сделайте что-нибудь из этого и проанализируйте эффект. Не стесняйтесь модифицировать схему — удалять по несколько фичей за шаг, параллелить и так далее, пункт времязатратный

In [ ]:
# (´-`）.｡oO( ... YOUR CODE HERE ... )

### Заключение и оценивание

Каждая из задач в ноутбуке имеет свою стоимость (указана в скобках рядом с задачей). При этом важно уточнить разницу между баллами за ноутбук и дополнительными баллами за позицию на приватном лидерборде в соревновании на Kaggle:

1. **Максимум за код/ноутбуки — 8.0 баллов.**
   То есть, независимо от суммарной теоретической суммы всех подпунктов в тексте задания, за реализацию в ноутбуке можно получить не более 8 баллов (6 за базу и 2 за продвинутый).

2. **Максимально возможная оценка за всю работу — 13.0 баллов.**
   Остальные до 5.0 баллов начисляются за результаты в соревновании на Kaggle (лидерборд), при выполненном и загруженном в систему Anytask ноутбуке.

Баллы за сореву состоят из трёх частей: трешхолды (до 2 баллов), процентильный бонус (до 2.0 баллов) и бонус за попадание в топ-10 (до 1.0 балла). Суммарный вклад соревнования не может превышать 5.0 баллов.

**A. Процентильный балл (не суммируется):**
* Если вы пробили трешхолд-9 (качество 0.34) — +1 балл.
* Если вы пробили трешхолд-10 (качество 0.36) — +2 балла.

**Б. Процентильный балл:**

* Если вы только прошли трешхолд-10, то баллов вы не получите.
* Если вы обогнали ≥ 10% участников, побивших трешхолд-10 — +0.5 балла.
* Если вы обогнали ≥ 30% участников — +1.0 балла.
* Если вы обогнали ≥ 60% участников — +1.5 балла.
* Если вы обогнали ≥ 90% участников (т.е. попали в топ 10%) — +2.0 балла.

**В. Балл за попадание в топ-10:**

* 1-е место — +1.00 балла
* 2-е–3-е место — +0.75 балла
* 4-е–6-е место — +0.50 балла
* 7-е–10-е место — +0.25 балла

Пример расчёта

* Вы сделали ноутбуки и получили за них 7.0 / 8.0.
* Вы, тем не менее, побили трешхолд-10 → +2.0
* На лидерборде вы, зайка, обогнали 10% участников, побивших трешхолд 10 → процентильный бонус +2.0.
* Ваша позиция — 3-е место → топ-10 бонус +0.75.
* Итого: 7.0 + 2.0 + 2.0 + 0.75 = 11.75.

Можете свериться с картинкой (левая граница не включительно)

<img src="https://i.postimg.cc/nhb25b42/newplot.png" height=720 width=1280>

**Требование к воспроизводимости**

Баллы за соревнование начисляются **только** при наличии пайплайна или ноутбука, который подтверждает результат лучшего сабмита. Такое решение нужно сдавать вместе с базовым и продвинутым ноутбуками и своим ников в kaggle в Anytask ассистенту. Он должен выполнять всё автоматически при запуске ноутбука: при последовательном исполнении всех ячеек ноутбука (без ручных вмешательств) он должен воспроизвести предобработку, обучение/инференс и сгенерировать итоговый CSV-файл с прогнозами, используемый для сабмита.